# Chapter 1 Lab — EEG Basics

**Think of it like this:** your brain is a huge stadium full of people (neurons) clapping (firing) at slightly different times. An EEG electrode is a microphone held *outside* the stadium — you don't hear any single clap, you hear the whole crowd's roar rising and falling. That roar is the squiggly line you're about to see.

Sometimes a seagull flies past the microphone and goes `SQUAWK` — way louder than the crowd, and it sounds completely different. In EEG, blinking or clenching your jaw is the seagull: a huge, obviously-not-brain signal called an **artifact**. Once you've heard the seagull once, you never mistake it for the crowd again.

**Goal of this lab:** load one real night of EEG, look at it, and learn to spot the seagulls.

## Setup
Run this once. `mne` is the standard Python library for EEG/MEG analysis — it's what loads the recording and draws it for you.

In [ ]:
!pip install -q mne
%matplotlib inline

## Download one real night of sleep EEG

We're using **Sleep-EDF**, a free public dataset on PhysioNet — real overnight recordings with a sleep expert's stage-by-stage labels already attached. MNE has a built-in downloader for it, so you don't have to touch PhysioNet directly.

Subject `0` is nicknamed "Alice" in MNE's own tutorials — we'll keep that name so it's easy to follow along with the official docs too.

In [ ]:
import mne
from mne.datasets.sleep_physionet.age import fetch_data

ALICE = 0
alice_files = fetch_data(subjects=[ALICE], recording=[1])[0]
# alice_files[0] = the PSG recording (.edf), alice_files[1] = the expert's sleep-stage labels
print(alice_files)

In [ ]:
raw = mne.io.read_raw_edf(
    alice_files[0],
    stim_channel="Event marker",
    infer_types=True,
    preload=True,
    verbose="error",
)
print(raw.info)
print("Channels:", raw.ch_names)
print("Sampling rate:", raw.info["sfreq"], "Hz")
print("Recording length:", raw.n_times / raw.info["sfreq"] / 3600, "hours")

**Sampling rate** is how many "snapshots" of the crowd's roar get recorded every second — this file uses 100 Hz, meaning 100 measurements per second per channel. That's the resolution your microphone is listening at.

## Look at the raw signal
This plots a 30-second window you can scroll through. Each row is a different channel — a different "microphone position" around the stadium (EEG channels, plus EOG for eye movement and EMG for muscle/chin tension).

In [ ]:
raw.plot(duration=30, n_channels=len(raw.ch_names), scalings="auto");

## Now go find a seagull

A real eyeblink artifact shows up as a big, sharp, slow up-down swing on the EOG (eye) channel — and often bleeds slightly into the frontal EEG channels too, since the eyes sit right next to the front of the brain.

Try different start times below (change `start_sec`) until you spot one. Blinks happen often while someone is still awake at the start of a recording, so try the first few minutes first.

In [ ]:
start_sec = 60  # try changing this
raw.plot(start=start_sec, duration=20, n_channels=len(raw.ch_names), scalings="auto");

## Done when...

You can look at a fresh 30-second window (try a few different `start_sec` values above) and confidently say **"that's a blink," "that's muscle noise," or "that's actual brain signal"** — without anyone telling you which is which.

That's it for Chapter 1. Next up: what the different *stages* of sleep actually look like on this same kind of trace.